# Previsão de `preco_venda` a partir do DW (AutoPrime)

Este notebook foi feito para **Google Colab**: lê o star schema (`Fato_Vendas_Carros` + dimensões) via MySQL e treina um modelo de **regressão** para estimar o preço de venda.

**Requisitos:** o MySQL do DW precisa aceitar conexões da internet (ex.: host na Aiven ou túnel). No Colab, use **Secrets** (ícone de chave) com as chaves indicadas abaixo — **não** coloque senhas em células públicas.

---

## 1. Instalar dependências

In [ ]:
%pip install -q pymysql sqlalchemy pandas scikit-learn

## 2. Credenciais do DW

No Colab: **Runtime → Secrets** (ou o ícone de chave) e crie:

| Secret | Exemplo |
|--------|--------|
| `DW_USER` | `engenheiro` (Docker) ou utilizador do teu servidor |
| `DW_PASSWORD` | palavra-passe |
| `DW_HOST` | hostname público do MySQL |
| `DW_PORT` | `3306` ou `13729` |
| `DW_NAME` | `dw_autoprime` |

Se correres fora do Colab, define as mesmas variáveis no ambiente ou edita os valores na célula seguinte (apenas localmente).

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine


def load_dw_config():
    """Lê credenciais dos Secrets do Colab ou do ambiente / fallback manual."""
    try:
        from google.colab import userdata
        return {
            "user": userdata.get("DW_USER"),
            "password": userdata.get("DW_PASSWORD"),
            "host": userdata.get("DW_HOST"),
            "port": userdata.get("DW_PORT"),
            "database": userdata.get("DW_NAME"),
        }
    except Exception:
        pass
    cfg = {
        "user": os.getenv("DW_USER", ""),
        "password": os.getenv("DW_PASSWORD", ""),
        "host": os.getenv("DW_HOST", ""),
        "port": os.getenv("DW_PORT", "3306"),
        "database": os.getenv("DW_NAME", "dw_autoprime"),
    }
    if not all([cfg["user"], cfg["password"], cfg["host"]]):
        raise RuntimeError(
            "Define os secrets DW_* no Colab ou variáveis de ambiente DW_USER, DW_PASSWORD, DW_HOST."
        )
    return cfg


cfg = load_dw_config()
u = quote_plus(str(cfg["user"]))
p = quote_plus(str(cfg["password"]))
url = f"mysql+pymysql://{u}:{p}@{cfg['host']}:{cfg['port']}/{cfg['database']}"
engine = create_engine(url, pool_pre_ping=True)
print("Engine criada (sem mostrar password).")

## 3. Carregar dados (mesma lógica do dashboard)

In [ ]:
QUERY_STAR = """
SELECT
    f.sk_venda,
    f.preco_venda,
    f.preco_mercado,
    f.quantidade_vendida,
    t.data_completa,
    t.ano,
    t.numero_mes,
    t.nome_mes,
    t.trimestre,
    t.semestre,
    t.dia_semana,
    t.indicador_fim_semana,
    l.nome_loja,
    l.regiao_loja,
    l.estado_loja,
    l.nome_estado_loja,
    v.marca,
    v.modelo,
    v.versao,
    v.categoria,
    v.transmissao,
    v.faixa_idade_veiculo
FROM Fato_Vendas_Carros f
JOIN Dim_Tempo_Venda t ON f.sk_tempo_venda = t.sk_tempo
JOIN Dim_Loja_Venda l ON f.sk_loja = l.sk_loja
JOIN Dim_Veiculo v ON f.sk_veiculo = v.sk_veiculo
"""

df = pd.read_sql(QUERY_STAR, engine)
print(df.shape)
df.head()

## 4. Preparar features e alvo

- **Alvo:** `preco_venda`.
- **Excluímos** `sk_venda`, `data_completa` e, no modelo principal, **`preco_mercado`** — assim o modelo aprende a partir de veículo, loja e tempo (sem “copiar” o preço de mercado). No final há uma célula opcional **com** `preco_mercado` para comparares o R².

In [ ]:
TARGET = "preco_venda"

drop_cols = {"sk_venda", "data_completa", TARGET, "preco_mercado"}
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[TARGET].astype(float)

# Booleanos e quantidades para tratamento numérico simples
if "indicador_fim_semana" in X.columns:
    X["indicador_fim_semana"] = X["indicador_fim_semana"].astype(int)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categóricas:", cat_cols)
print("Numéricas:", num_cols)

## 5. Treino / teste e pipeline (Random Forest)

Uso de `OrdinalEncoder` para texto + árvores: evita explosão de colunas com one-hot em `modelo`/`versão`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        (
            "enc",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

num_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([("prep", preprocess), ("rf", model)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
r2 = r2_score(y_test, pred)

print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

## 6. Importância aproximada das features (Random Forest)

In [ ]:
rf = pipe.named_steps["rf"]
prep = pipe.named_steps["prep"]
names = prep.get_feature_names_out()
imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)
imp.head(15)

## 7. (Opcional) Modelo com `preco_mercado` como feature

Espera-se **R² maior**: o preço de mercado é um forte proxy do preço realizado.

In [ ]:
drop_b = {"sk_venda", "data_completa", TARGET}
Xb = df[[c for c in df.columns if c not in drop_b]].copy()
if "indicador_fim_semana" in Xb.columns:
    Xb["indicador_fim_semana"] = Xb["indicador_fim_semana"].astype(int)

cat_b = Xb.select_dtypes(include=["object", "category"]).columns.tolist()
num_b = [c for c in Xb.columns if c not in cat_b]

preprocess_b = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_b),
        ("num", num_pipe, num_b),
    ]
)
pipe_b = Pipeline(
    [
        ("prep", preprocess_b),
        ("rf", RandomForestRegressor(
            n_estimators=200,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

X_tr, X_te, y_tr, y_te = train_test_split(Xb, y, test_size=0.2, random_state=42)
pipe_b.fit(X_tr, y_tr)
pred_b = pipe_b.predict(X_te)
print("Com preco_mercado — R²:", r2_score(y_te, pred_b))
print("Com preco_mercado — MAE:", mean_absolute_error(y_te, pred_b))

---

### Notas

- Para previsão **no tempo**, prefere dividir por `data_completa` (treino passado / teste futuro) em vez de `train_test_split` aleatório.
- Garante na Aiven/firewall que o IP do Colab pode mudar; pode ser preciso **allow list** larga ou outro método de acesso.
- **Não** faças upload deste notebook para repositório público com credenciais nas células.